<a href="https://colab.research.google.com/github/Vitalik-Hakim/lab4/blob/main/lab_4.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Lab 4: LLMs and Prompt Engineering for Decision Support

**Duration:** 2 weeks [30 Jul - 13 Aug, 2026]
**Due Date:** 13th August, 2026
**Format:** Jupyter Notebook / Google Colab + external APIs + GitHub version control
**Grading:** This is a graded lab.

**Student Name:** [Enter Name]
**Student ID:** [Enter ID]

---

### Objective

In the previous labs you *trained* models. In this lab you will *use* a model that someone
else spent millions of dollars training — a **Large Language Model (LLM)** — and learn that
getting good results out of one is an engineering discipline of its own: **prompt
engineering**.

You will build a **decision support system for a microfinance loan officer**. Given a pile of
free-text loan application letters, your system will:

1. **Summarize** each application into a short, factual brief,
2. **Extract** specific structured data points (JSON) that a downstream system could store,
3. Produce a **decision-support recommendation** — while keeping the human firmly in the loop.

Just as importantly, you will **evaluate** the LLM's output for quality, reliability, and
appropriateness: Does it hallucinate? Is it consistent across runs? Should it be trusted to
make the final call?

---

### Choosing an API provider

You need an LLM API with a **free tier**. Recommended options (pick ONE):

| Provider | Free tier | Notes |
|---|---|---|
| **Groq** (recommended) | Yes, generous | OpenAI-compatible API, very fast, open models (Llama) |
| **Google Gemini** | Yes | `google-generativeai` package |
| **Hugging Face Inference API** | Yes, limited | Many open models |
| OpenAI / Anthropic | Paid | Fine if you already have credits |

The notebook's example code uses the **OpenAI-compatible chat format** (works with Groq and
OpenAI directly; Gemini users adapt the call in one place). Everything else in the lab is
provider-agnostic.

---
### Part 0: Repository and API-key setup

1. Create a **public** repository named `lab-4-llm-decision-support` and save this notebook
   inside it.
2. Sign up with your chosen provider and create an **API key**.
3. **NEVER hard-code or commit your API key.** This is a graded requirement.
   - Locally: put it in a `.env` file and add `.env` to `.gitignore`.
   - Colab: use the Secrets panel (key icon) and read it with `google.colab.userdata`.
4. Add a `requirements.txt`: `openai python-dotenv pandas matplotlib`.
5. Commit and push after **each Part** — we will check for incremental commits.

> **A leaked key in your commit history = resubmission + penalty.** Keys can be scraped from
> public repos within minutes.

In [33]:
# API-key setup — DO NOT hard-code your key in this cell.

import os
import pandas as pd
import json
import re

# --- Local (with a .env file) ---
# from dotenv import load_dotenv
# load_dotenv()
# API_KEY = os.environ["GROQ_API_KEY"]

# --- Google Colab (Secrets panel) ---
from google.colab import userdata
API_KEY = userdata.get("GROQ_API_KEY")
# TODO: set API_KEY using ONE of the methods above.

# OpenAI-compatible client (works for Groq and OpenAI; Gemini users see their docs):
from openai import OpenAI

client = OpenAI(
    api_key=API_KEY,
    base_url="https://api.groq.com/openai/v1",   # remove this line if using OpenAI itself
)
MODEL = "llama-3.3-70b-versatile"                # or your provider's model name

print("Client ready.")

Client ready.


---
# Section 1 — Talking to an LLM Programmatically

Before building anything, understand the anatomy of an API call: **messages and roles**
(`system`, `user`, `assistant`), and the **generation parameters** (`temperature`,
`max_tokens`).

### Part 1.1 — Your first API call

In [34]:
# TODO: Write a helper function you will reuse for the WHOLE lab:
#
def ask_llm(user_prompt, system_prompt="You are a helpful assistant.",
            temperature=0.7, max_tokens=500):
    response = client.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user",   "content": user_prompt},
        ],
        temperature=temperature,
        max_tokens=max_tokens,
    )
    return response
#
# TODO: Call it once with a simple question and print the answer.
response = ask_llm("Who colonized ghana?")
# TODO: Print response.usage as well — how many tokens did your call consume?

print("Answer:")
print(response.choices[0].message.content)

print("\nToken usage:")
print(response.usage)

Answer:
Ghana was colonized by the British. The British colonized the Gold Coast, which is now known as Ghana, in the late 19th century. The British established a colony on the Gold Coast in 1844, and it remained a British colony until Ghana gained independence on March 6, 1957.

Before the British, the Portuguese, Dutch, and other European powers had established trading posts and forts on the Gold Coast, but the British eventually became the dominant colonial power in the region. The British colonization of Ghana was marked by the exploitation of the country's natural resources, including gold, and the imposition of British culture, language, and institutions on the local population.

The British also established a system of indirect rule, where local chiefs and traditional leaders were used to administer the colony on behalf of the British. This system helped to maintain British control while minimizing the need for direct British administration.

Ghana's struggle for independence wa

**Student Reasoning — Anatomy of a call**
*1. What is the difference between the `system` and `user` roles? Give an example of
something that belongs in each.*
*2. What is a token, roughly? Why do API providers bill per token rather than per request?*

> **Answer:** [Double-click to edit]
1.
System and User roles:
The system role defines how the LLM should behave, while the user role contains the actual request.
An example is:
System: "You are a helpful loan officer assistant."
User: "Summarize this loan application."

2.
Tokens:
A token is a small piece of text, such as part of a word or a short word. Providers like Groq bill per token because the model's computational work depends largely on the amount of text processed and generated, not simply the number of requests.

### Part 1.2 — Temperature: the randomness dial

In [35]:
# TODO: Ask the SAME question 5 times at temperature=0.0 and 5 times at temperature=1.2.
#   A good test question: "Suggest a name for a savings product for market traders in Accra."

# TODO: Print all 10 answers, grouped by temperature.

question = "Suggest a name for a savings product for market traders in Accra."

# Temperature = 0.0
print("Temperature: 0")
for i in range(5):
    response = ask_llm(question, temperature=0.0)
    print(f"{i+1}. {response.choices[0].message.content}")

print("\n")
# Temperature = 1.2
print("Temperature: 1.2")
for i in range(5):
    response = ask_llm(question, temperature=1.2)
    print(f"{i+1}. {response.choices[0].message.content}")


Temperature: 0
1. Here are a few suggestions for a savings product for market traders in Accra:

1. **Makola Save**: "Makola" is a well-known market in Accra, so this name could resonate with market traders.
2. **Trader's Treasure**: This name emphasizes the idea of saving and accumulating wealth.
3. **Sika Kokoo**: "Sika" means "money" in the Akan language, and "Kokoo" means "gather" or "collect". This name could appeal to market traders who want to gather and save their earnings.
4. **Market Mobi**: This name is short and catchy, and "Mobi" implies mobility and flexibility, which could be attractive to market traders who need to access their savings easily.
5. **Adanfo Save**: "Adanfo" means "friends" or "partners" in the Akan language, which could convey a sense of community and cooperation among market traders.
6. **Kae Dwa**: "Kae Dwa" means "good savings" or "profitable savings" in the Akan language, which could appeal to market traders who want to save and grow their money.
7. *

**Student Reasoning — Temperature**
*What did you observe at each temperature? For the loan decision-support system you are about
to build, which temperature regime is appropriate, and why?*

> **Answer:** [Double-click to edit]
At temperature 0.0, the answers were mostly identical or very similar, showing consistent and predictable outputs. At temperature 1.2, the answers were more varied and creative.

For the loan decision support system, temperature 0.0 is more appropriate because we want consistent, factual, and reliable outputs rather than creative or random responses.

---
# Section 2 — The Dataset: Loan Application Letters

Run the next cell to load **six loan application letters** submitted to a (fictional)
microfinance institution in Ghana, plus **gold-standard extraction labels** for three of them
(you will use these for evaluation in Section 4).

Read at least two letters fully before moving on — you cannot engineer prompts for text you
have not read.

In [36]:
LETTERS = {
"L001": """Dear Sir/Madam,
My name is Akosua Mensah and I have been selling provisions at Makola Market for 12 years.
I am applying for a loan of GHS 8,000 to buy a deep freezer and expand into frozen foods.
My current stall makes about GHS 900 profit each month. I have saved GHS 2,500 with your
susu scheme over the past two years and I have never missed a contribution. I can repay
GHS 450 monthly over 20 months. My sister, a teacher, will stand as my guarantor.
Thank you for considering my application.""",

"L002": """Hello,
I am Kwame Boateng, a commercial driver in Kumasi. I need GHS 25,000 urgently to repair my
trotro engine and settle some personal debts. Business has been slow but it will surely
pick up after the festive season. I can pay back whenever the money comes. I do not have
collateral at the moment but God willing everything will be fine. Please help me quickly.""",

"L003": """Dear Loan Committee,
I am Efua Darko, owner of Darko Fashions, a registered dressmaking business in Takoradi
(registration no. BN-2019-4482). I employ three apprentices. I request GHS 15,000 to
purchase two industrial sewing machines and fabric stock ahead of the Christmas season.
Last year my December revenue alone was GHS 22,000; monthly profit averages GHS 2,800.
I hold a fixed deposit of GHS 5,000 with GCB which I can pledge. Proposed repayment:
GHS 1,100 monthly for 15 months. Attached are my sales records for the past 18 months.""",

"L004": """Good day,
My name is Yaw Owusu. I want a loan for my poultry farm at Nsawam. The amount is GHS 12,000
for feed and 500 new layers. I started the farm last year. Sometimes I make good money,
around GHS 1,500 in a good month, but bird flu affected us in March and I lost many birds.
I am rebuilding now. I can repay in 18 months. My uncle has agreed to guarantee the loan
with his taxi.""",

"L005": """Dear Manager,
I am writing on behalf of the Adenta Women's Weaving Cooperative (14 members). We seek
GHS 30,000 to buy a bulk order of yarn directly from the factory, cutting out middlemen and
raising our margins from 15% to about 35%. The cooperative has operated for 6 years and
holds GHS 9,000 in our group account. We propose repayment of GHS 2,000 monthly over
16 months, backed by our group savings and joint liability agreement.""",

"L006": """Hi,
This is Kofi. I saw your advert. I want GHS 50,000 to start a car washing business, a
provision shop, and also import phones from Dubai. I am 22 and full of energy. I have not
started any of these yet but my friends say I am very business minded. I will pay back in
one year when the businesses are booming. No collateral but I am trustworthy.""",
}

# Gold-standard labels for three letters (for Section 4 evaluation):
GOLD = {
  "L001": {"applicant_name": "Akosua Mensah", "amount_ghs": 8000,  "purpose": "buy deep freezer / expand into frozen foods",
           "monthly_profit_ghs": 900,  "has_collateral_or_guarantor": True,  "repayment_months": 20},
  "L003": {"applicant_name": "Efua Darko",    "amount_ghs": 15000, "purpose": "industrial sewing machines and fabric stock",
           "monthly_profit_ghs": 2800, "has_collateral_or_guarantor": True,  "repayment_months": 15},
  "L006": {"applicant_name": "Kofi",          "amount_ghs": 50000, "purpose": "car wash, provision shop, phone imports",
           "monthly_profit_ghs": None, "has_collateral_or_guarantor": False, "repayment_months": 12},
}

print(f"{len(LETTERS)} letters loaded.")

6 letters loaded.


---
# Section 3 — Prompt Engineering for the Decision Support System

You will now build the three components of the system, iterating on your prompts as you go.
**Keep every major prompt version** — Section 3.4 asks you to commit your prompt templates
and document how they evolved.

### Part 3.1 — Component 1: Summarization
Turn a rambling letter into a 3-4 sentence factual brief a busy loan officer can scan.

In [37]:
# TODO: Write SUMMARY_PROMPT_V1 — your first, naive attempt (e.g. just "Summarize this:").
#   Run it on L002 and L006. Read the output critically.

SUMMARY_PROMPT_V1 = "Summarize this:"

v1_l002 = ask_llm(
    f"{SUMMARY_PROMPT_V1}\n\n{LETTERS['L002']}"
)

v1_l006 = ask_llm(
    f"{SUMMARY_PROMPT_V1}\n\n{LETTERS['L006']}"
)

print("V1 - L002")
print(v1_l002.choices[0].message.content)
print("\n")
print("V1 - L006 ")
print(v1_l006.choices[0].message.content)

# TODO: Now write SUMMARY_PROMPT_V2 as a proper template with:
#   - a system prompt giving the LLM a ROLE (e.g. "You are an assistant to a microfinance
#     loan officer...") and constraints (factual, neutral, no invented details, 3-4 sentences)
#   - a user prompt template like: f"Summarize this loan application:\n\n{letter_text}"
#   Run V2 on the same two letters at temperature=0.

SUMMARY_SYSTEM_V2 = """You are an assistant to a microfinance loan officer.
Summarize loan applications in a factual and neutral manner.
Do not invent or assume any information that is not stated in the application.
Keep the summary to 3-4 sentences."""

SUMMARY_PROMPT_V2 = "Summarize this loan application:\n\n{letter_text}"

v2_l002 = ask_llm(
    SUMMARY_PROMPT_V2.format(letter_text=LETTERS["L002"]),
    system_prompt=SUMMARY_SYSTEM_V2,
    temperature=0
)

v2_l006 = ask_llm(
    SUMMARY_PROMPT_V2.format(letter_text=LETTERS["L006"]),
    system_prompt=SUMMARY_SYSTEM_V2,
    temperature=0
)
# TODO: Compare V1 vs V2 outputs side by side. Keep both prompt versions in this notebook.
print("\n" + "=" * 60)
print("L002 - V1 vs V2")
print("=" * 60)

print("\n--- V1 ---")
print(v1_l002.choices[0].message.content)

print("\n--- V2 ---")
print(v2_l002.choices[0].message.content)


print("\n" + "=" * 60)
print("L006 - V1 vs V2")
print("=" * 60)

print("\n--- V1 ---")
print(v1_l006.choices[0].message.content)

print("\n--- V2 ---")
print(v2_l006.choices[0].message.content)

V1 - L002
Kwame Boateng, a commercial driver in Kumasi, is urgently seeking GHS 25,000 to repair his vehicle's engine and settle personal debts. He's experiencing a slow business period but expects it to improve after the festive season and is willing to repay the loan when he can, despite not having collateral at the moment.


V1 - L006 
Kofi, a 22-year-old, is seeking a loan of GHS 50,000 to start three businesses: a car washing service, a provision shop, and a phone import business from Dubai. He has no prior experience or collateral, but claims to be "business-minded" and promises to repay the loan within a year when his businesses become successful, relying on his trustworthiness as assurance.

L002 - V1 vs V2

--- V1 ---
Kwame Boateng, a commercial driver in Kumasi, is urgently seeking GHS 25,000 to repair his vehicle's engine and settle personal debts. He's experiencing a slow business period but expects it to improve after the festive season and is willing to repay the loan whe

**Student Reasoning — Summarization prompts**
*1. What concrete problems did V1's output have that V2 fixed? Quote examples.*
*2. Why is "no invented details" an essential instruction in this application? What is this
failure mode called in the LLM literature?*

> **Answer:** [Double-click to edit]
V1 could be too general and may include unnecessary details or less focused wording. V2 produces a more focused, factual 3-4 sentence summary and avoids assumptions.
“No invented details” is essential because loan decisions must be based only on information provided by the applicant. When an LLM generates unsupported or false information, this is called hallucination.

### Part 3.2 — Component 2: Structured extraction (JSON)
Downstream software cannot read prose. Extract the fields in `GOLD` as strict JSON.

In [38]:
# TODO: Write EXTRACT_PROMPT — a template that instructs the model to return ONLY a JSON
#   object with EXACTLY these keys:
#     applicant_name (string), amount_ghs (number), purpose (string),
#     monthly_profit_ghs (number or null), has_collateral_or_guarantor (boolean),
#     repayment_months (number or null)
#   Techniques to use:
#     - explicit schema in the prompt
#     - ONE worked example (few-shot) using a letter you write yourself (not from LETTERS!)
#     - "If a field is not stated in the letter, use null. Do not guess."
#     - temperature=0

EXTRACT_PROMPT = """
Extract information from the loan application below.

Return ONLY a valid JSON object with EXACTLY these keys:
{{
  "applicant_name": "string",
  "amount_ghs": 0,
  "purpose": "string",
  "monthly_profit_ghs": 0,
  "has_collateral_or_guarantor": true,
  "repayment_months": 0
}}

Rules:
- applicant_name must be a string.
- amount_ghs must be a number.
- purpose must be a string.
- monthly_profit_ghs must be a number or null.
- has_collateral_or_guarantor must be true or false.
- repayment_months must be a number or null.
- If a field is not stated in the letter, use null. Do not guess.
- Return ONLY JSON. Do not include explanations or markdown.

Worked example:

Letter:
"My name is Ama Osei. I run a small bakery in Accra and need GHS 6,000
to purchase an oven. I make about GHS 1,200 profit per month. My brother
will guarantee the loan. I can repay over 10 months."

JSON:
{{
  "applicant_name": "Ama Osei",
  "amount_ghs": 6000,
  "purpose": "purchase an oven",
  "monthly_profit_ghs": 1200,
  "has_collateral_or_guarantor": true,
  "repayment_months": 10
}}

Now extract the information from this loan application:

{letter_text}
"""


# TODO: Write extract_fields(letter_text) that calls the LLM, strips any ```json fences,
#   json.loads() the result, and returns a dict. Handle parse failures gracefully
#   (return None and print a warning).


def extract_fields(letter_text):
    response = ask_llm(
        EXTRACT_PROMPT.format(letter_text=letter_text),
        temperature=0,
        max_tokens=500
    )

    result = response.choices[0].message.content.strip()

    # Remove markdown JSON fences if the model adds them
    result = re.sub(r"^```json\s*", "", result)
    result = re.sub(r"\s*```$", "", result)

    try:
        return json.loads(result)
    except json.JSONDecodeError:
        print("Warning: Failed to parse LLM response as JSON.")
        print("Raw response:", result)
        return None


# TODO: Run it on ALL SIX letters; collect results into a pandas DataFrame (one row per
#   letter) and display it.


results = []

for letter_id, letter_text in LETTERS.items():
    extracted = extract_fields(letter_text)

    if extracted is not None:
        extracted["letter_id"] = letter_id
        results.append(extracted)
    else:
        results.append({
            "letter_id": letter_id,
            "applicant_name": None,
            "amount_ghs": None,
            "purpose": None,
            "monthly_profit_ghs": None,
            "has_collateral_or_guarantor": None,
            "repayment_months": None
        })

df_extracted = pd.DataFrame(results)

columns = [
    "letter_id",
    "applicant_name",
    "amount_ghs",
    "purpose",
    "monthly_profit_ghs",
    "has_collateral_or_guarantor",
    "repayment_months"
]

df_extracted = df_extracted[columns]

display(df_extracted)

,letter_id,applicant_name,amount_ghs,purpose,monthly_profit_ghs,has_collateral_or_guarantor,repayment_months
0,L001,Akosua Mensah,8000,buy a deep freezer and expand into frozen foods,900.0,True,20.0
1,L002,Kwame Boateng,25000,repair my trotro engine and settle some person...,NaN,False,NaN
2,L003,Efua Darko,15000,purchase two industrial sewing machines and fa...,2800.0,True,15.0
3,L004,Yaw Owusu,12000,for feed and 500 new layers,1500.0,True,18.0
4,L005,Adenta Women's Weaving Cooperative,30000,buy a bulk order of yarn,NaN,True,16.0
5,L006,Kofi,50000,"start a car washing business, a provision shop...",NaN,False,12.0


**Student Reasoning — Structured extraction**
*1. Why must the few-shot example NOT come from the six letters you are processing?*
*2. Why "use null, do not guess" — what did the model do without that instruction?*
*3. Why is temperature=0 the right choice for extraction but arguably not for creative tasks?*

> **Answer:** [Double-click to edit]

1. The example must be separate so it doesn't leak information from the six test letters into the model's extraction. Otherwise, it could make the evaluation unfair.

2. “Use null, do not guess” prevents the model from making up missing information. Without it, the model may infer or invent values for fields that are not stated, which is a form of hallucination.

3. Temperature 0 gives more consistent and predictable outputs, which is important for structured extraction. Creative tasks benefit from higher temperatures because they need more variety and originality.

### Part 3.3 — Component 3: The decision-support brief
Combine everything: for each letter, produce a recommendation brief for the loan officer —
strengths, risks, missing information, and a suggested next step. The system must
**support** the decision, not **make** it.

In [39]:
# TODO: Write BRIEF_PROMPT — it receives the letter AND your extracted JSON, and must output:
#     1. Strengths (bullet points, grounded in the letter)
#     2. Risks / red flags (bullet points)
#     3. Missing information the officer should request
#     4. Suggested next step (e.g. "invite for interview", "request documents",
#        "flag for senior review") — NOT "approve" or "reject".
#   Give the model an explicit instruction that final decisions are made by humans.

BRIEF_PROMPT = """
You are an assistant supporting a microfinance loan officer.

Review the loan application and the extracted information provided below.

Create a decision-support brief with exactly these sections:

1. Strengths
- List strengths grounded only in the information stated in the letter.

2. Risks / Red Flags
- List potential risks or concerns grounded only in the letter.
- Do not invent or assume information.

3. Missing Information
- List important information or documents the loan officer should request
  before making a decision.
- If no important information is missing, state that clearly.

4. Suggested Next Step
- Suggest an appropriate next step such as inviting the applicant for an
  interview, requesting documents, or flagging the application for senior review.
- Do NOT recommend "approve" or "reject".

The final loan decision must always be made by a human loan officer.
The purpose of this brief is to support the human decision-maker, not replace them.

Loan application:
{letter_text}

Extracted information:
{extracted_json}
"""


# TODO: Generate briefs for ALL SIX letters. Print the briefs for L001, L002, and L006 —
#   three very different applications.

briefs = {}

for letter_id, letter_text in LETTERS.items():
    extracted = df_extracted[
        df_extracted["letter_id"] == letter_id
    ].iloc[0].to_dict()

    extracted_json = json.dumps(extracted, indent=2)

    response = ask_llm(
        BRIEF_PROMPT.format(
            letter_text=letter_text,
            extracted_json=extracted_json
        ),
        temperature=0,
        max_tokens=700
    )

    briefs[letter_id] = response.choices[0].message.content



print("=" * 60)
print("L001 — DECISION-SUPPORT BRIEF")
print("=" * 60)
print(briefs["L001"])

print("\n" + "=" * 60)
print("L002 — DECISION-SUPPORT BRIEF")
print("=" * 60)
print(briefs["L002"])
# print("Total tokens used for this application")
# print(response.usage)

print("\n" + "=" * 60)
print("L006 — DECISION-SUPPORT BRIEF")
print("=" * 60)
print(briefs["L006"])

L001 — DECISION-SUPPORT BRIEF
## 1. Strengths
- The applicant, Akosua Mensah, has 12 years of experience selling provisions at Makola Market, indicating stability and knowledge in her business.
- She has a steady monthly profit of GHS 900, which suggests a viable business operation.
- Akosua has saved GHS 2,500 through the susu scheme over two years without missing a contribution, demonstrating her ability to save and commit to financial obligations.
- She has a guarantor, her sister, who is a teacher, potentially providing an additional layer of financial security.
- The applicant has a clear plan for loan repayment, proposing to pay GHS 450 monthly over 20 months.

## 2. Risks / Red Flags
- The loan amount of GHS 8,000 is significant compared to the applicant's monthly profit and savings, which might pose a risk if the business expansion does not generate enough additional income to cover the loan repayments.
- There is no detailed information provided about the sister's financial si

**Student Reasoning — Decision support**
*1. Compare the briefs for L003 (strong application) and L006 (weak application). Did the
system identify the right strengths and red flags in each?*
*2. Why did we forbid the model from outputting "approve"/"reject"? Give one practical and
one ethical reason.*

> **Answer:** [Double-click to edit]

1. L003 had clear strengths: an established registered business, existing profit, sales records, collateral, and a specific repayment plan. L006 had major red flags: no existing business, no collateral, multiple unstarted business ideas, and an uncertain repayment plan. The system should identify these differences correctly.

2. Practical reason: The LLM may lack important financial information needed for a final decision, so a human officer must review the application.
Ethical reason: Loan approval/rejection can significantly affect someone's livelihood, so the final decision should remain with a human rather than an AI.

### Part 3.4 — Commit your prompt templates
Prompts ARE code. Save your final `SUMMARY_PROMPT`, `EXTRACT_PROMPT`, and `BRIEF_PROMPT` into
a separate file `prompts.py` (or `prompts.md`) in your repository and commit it with a
message describing how the prompts evolved. Paste your commit hash below.

> **Commit hash:** 429b8ab789c35115940a33d6ab788e15ee846728

---
# Section 4 — Evaluation: Quality, Reliability, Appropriateness

An impressive demo is not a trustworthy system. Now measure it.

### Part 4.1 — Extraction accuracy against gold labels

In [40]:
# TODO: For the three letters in GOLD, compare your extracted DataFrame to the gold values
#   field by field. Compute per-field accuracy across the three letters
#   (name matching can be case-insensitive; numbers must match exactly).

# TODO: Display a small table: rows = fields, columns = L001 / L003 / L006 / accuracy.

fields = [
    "applicant_name",
    "amount_ghs",
    "purpose",
    "monthly_profit_ghs",
    "has_collateral_or_guarantor",
    "repayment_months"
]

evaluation_rows = []

for field in fields:
    row = {"field": field}
    correct_count = 0

    for letter_id in GOLD:
        predicted = df_extracted.loc[
            df_extracted["letter_id"] == letter_id, field
        ].iloc[0]

        actual = GOLD[letter_id][field]

        # Names: case-insensitive comparison
        if field == "applicant_name":
            is_correct = str(predicted).strip().lower() == str(actual).strip().lower()

        # All other fields: exact matching
        else:
            is_correct = predicted == actual

        row[letter_id] = "✓" if is_correct else "✗"

        if is_correct:
            correct_count += 1

    row["accuracy"] = f"{correct_count / len(GOLD):.1%}"
    evaluation_rows.append(row)

evaluation_df = pd.DataFrame(evaluation_rows)

evaluation_df = evaluation_df[
    ["field", "L001", "L003", "L006", "accuracy"]
]

display(evaluation_df)

,field,L001,L003,L006,accuracy
0,applicant_name,✓,✓,✓,100.0%
1,amount_ghs,✓,✓,✓,100.0%
2,purpose,✗,✗,✗,0.0%
3,monthly_profit_ghs,✓,✓,✗,66.7%
4,has_collateral_or_guarantor,✓,✓,✓,100.0%
5,repayment_months,✓,✓,✓,100.0%


### Part 4.2 — Reliability: is the system consistent?

In [41]:
# TODO: Run extract_fields() on letter L004 FIVE times at temperature=0 and FIVE times at
#   temperature=1.0.

# TODO: For each temperature, report how many of the 5 runs produced (a) valid JSON and
#   (b) identical values across runs. A simple approach: json.dumps(result, sort_keys=True)
#   and count unique strings.
SUMMARY_SYSTEM_PROMPT = """
You are an assistant to a microfinance loan officer.
Summarize loan applications in a factual and neutral manner.
Do not invent or assume any information that is not stated in the application.
Keep the summary to 3-4 sentences.
"""

# Update extract_fields() so temperature can be changed for the experiment
def extract_fields(letter_text, temperature=0):
    response = ask_llm(
        EXTRACT_PROMPT.format(letter_text=letter_text),
        temperature=temperature,
        max_tokens=500
    )

    result = response.choices[0].message.content.strip()

    # Remove markdown JSON fences if the model adds them
    result = re.sub(r"^```json\s*", "", result)
    result = re.sub(r"\s*```$", "", result)

    try:
        return json.loads(result)
    except json.JSONDecodeError:
        print("Warning: Failed to parse LLM response as JSON.")
        print("Raw response:", result)
        return None


# Run L004 five times at each temperature
results_by_temperature = {}

for temperature in [0, 1.0]:
    results = []

    for i in range(5):
        result = extract_fields(LETTERS["L004"], temperature=temperature)
        results.append(result)

    results_by_temperature[temperature] = results


# Report validity and consistency
for temperature, results in results_by_temperature.items():

    valid_results = [result for result in results if result is not None]

    # Convert valid dictionaries to sorted JSON strings for comparison
    unique_results = set(
        json.dumps(result, sort_keys=True)
        for result in valid_results
    )

    print(f"\n=== Temperature {temperature} ===")
    print(f"Valid JSON: {len(valid_results)}/5")
    print(f"Unique valid outputs: {len(unique_results)}")
    print(f"Identical across all 5 runs: {len(unique_results) == 1 and len(valid_results) == 5}")

    print("\nOutputs:")
    for i, result in enumerate(results, 1):
        print(f"Run {i}: {result}")


=== Temperature 0 ===
Valid JSON: 5/5
Unique valid outputs: 1
Identical across all 5 runs: True

Outputs:
Run 1: {'applicant_name': 'Yaw Owusu', 'amount_ghs': 12000, 'purpose': 'for feed and 500 new layers', 'monthly_profit_ghs': 1500, 'has_collateral_or_guarantor': True, 'repayment_months': 18}
Run 2: {'applicant_name': 'Yaw Owusu', 'amount_ghs': 12000, 'purpose': 'for feed and 500 new layers', 'monthly_profit_ghs': 1500, 'has_collateral_or_guarantor': True, 'repayment_months': 18}
Run 3: {'applicant_name': 'Yaw Owusu', 'amount_ghs': 12000, 'purpose': 'for feed and 500 new layers', 'monthly_profit_ghs': 1500, 'has_collateral_or_guarantor': True, 'repayment_months': 18}
Run 4: {'applicant_name': 'Yaw Owusu', 'amount_ghs': 12000, 'purpose': 'for feed and 500 new layers', 'monthly_profit_ghs': 1500, 'has_collateral_or_guarantor': True, 'repayment_months': 18}
Run 5: {'applicant_name': 'Yaw Owusu', 'amount_ghs': 12000, 'purpose': 'for feed and 500 new layers', 'monthly_profit_ghs': 1500,

### Part 4.3 — Hallucination probing

In [42]:
# TODO: Design TWO adversarial tests and run them:
#   Test 1 — Ask your summarizer a question about a detail that is NOT in a letter
#     (e.g. "What is the applicant's credit score?"). Does it admit the information is
#     absent, or does it invent one?
#   Test 2 — Feed your extractor an EMPTY or IRRELEVANT text (e.g. a weather report).
#     Does it return nulls, or does it fabricate an applicant?

# TODO: Record the outputs verbatim below and label each PASS or FAIL.
SUMMARY_SYSTEM_PROMPT = """
You are an assistant to a microfinance loan officer.
Summarize loan applications in a factual and neutral manner.
Do not invent or assume any information that is not stated in the application.
Keep the summary to 3-4 sentences.
"""

# test1 missing details
test1_question = f"""
What is the applicant's credit score?

Loan application:
{LETTERS["L002"]}
"""

test1_response = ask_llm(
    test1_question,
    system_prompt=SUMMARY_SYSTEM_PROMPT,
    temperature=0
)

test1_output = test1_response.choices[0].message.content

print("TEST 1 — Missing Information")
print(test1_output)

# PASS if the model states that the credit score is not provided.
# FAIL if it invents or assumes a credit score.
# it passed
print("\nResult:", "PASS")


#test2 — Irrelevant input
weather_report = """
Today's weather forecast: Accra will be partly cloudy with temperatures
between 25 and 31 degrees Celsius. There is a 30% chance of rain in the
afternoon. Winds will be moderate.
"""

test2_output = extract_fields(weather_report)

print("\nTEST 2 — Irrelevant Input")
print(test2_output)

# PASS if the extractor does not fabricate an applicant.
# Ideally, fields that cannot be determined should be null.
test2_pass = (
    test2_output is not None
    and test2_output.get("applicant_name") is None
    and test2_output.get("amount_ghs") is None
    and test2_output.get("purpose") is None
    and test2_output.get("monthly_profit_ghs") is None
    and test2_output.get("has_collateral_or_guarantor") is None
    and test2_output.get("repayment_months") is None
)

print("\nResult:", "PASS" if test2_pass else "FAIL")

TEST 1 — Missing Information
The loan application is from Kwame Boateng, a commercial driver in Kumasi, who is requesting GHS 25,000 to repair his trotro engine and settle personal debts. The applicant mentions that business has been slow, but expects it to improve after the festive season. There is no mention of the applicant's credit score in the application. The applicant also states that they do not have collateral to offer at the moment.

Result: PASS

TEST 2 — Irrelevant Input
{'applicant_name': None, 'amount_ghs': None, 'purpose': None, 'monthly_profit_ghs': None, 'has_collateral_or_guarantor': None, 'repayment_months': None}

Result: PASS


**Student Reasoning — Evaluation results**
*1. Report your extraction accuracy. Which field was hardest for the model and why?*
*2. What did the reliability experiment show about temperature and production systems?*
*3. Did your system hallucinate under probing? If yes, how could the prompt (or the system
design around it) reduce the risk?*

> **Answer:** [Double-click to edit]
1. Extraction accuracy: The model achieved high accuracy overall. The hardest field was purpose, because applicants sometimes describe multiple purposes or phrase them differently, making exact matching more difficult.
2. Reliability: Temperature 0 produced more consistent outputs across repeated runs, while higher temperature introduced more variation. This shows that production extraction systems should use low temperature for predictable results.
3. Hallucination: If hallucinations occurred during probing, the prompt can reduce them by explicitly saying “use null, do not guess” and requiring strict JSON. The system can also validate outputs and send uncertain or invalid cases to a human reviewer rather than automatically trusting the LLM.

### Part 4.4 — Appropriateness: should this system exist?
No code in this part — just judgment, which is the scarcest skill in AI for business.

**Student Reasoning — Appropriateness**
*1. Letters L002 and L006 would likely be declined. If the bank fully automated decisions
with your system, who could be unfairly harmed, and how? Consider applicants who write
poorly in English but run solid businesses.*
*2. Loan letters contain personal data. What are the implications of sending them to a
third-party API in another country? What would you check before deploying this at a real
Ghanaian microfinance institution?*
*3. Name TWO concrete safeguards you would build around this system in production (think:
human review points, logging, appeal processes, monitoring).*

> **Answer:** [Double-click to edit]

1. Unfair harm: Applicants who write poorly in English could be judged as less credible even if they operate strong businesses. This could disadvantage people based on writing ability rather than actual creditworthiness.

2. Thirdparty API: Personal and financial information would be sent to an external company and potentially processed outside Ghana, which could be dangerous. Before I deploy, I would check data protection/privacy requirements and policies, where the data is stored, how it is used, retention policies, security that protects the data, and whether the institution has the necessary consent and agreements to store this kind of sensitive.

3. Two safeguards can be implemented:
Human review: No loan is automatically approved or rejected; a loan officer reviews the AI's recommendation and then adds a human layer of thinking to it, whether to approve or deny.
Monitoring and audit logs: Record AI outputs and decisions, regularly check for errors or unfair patterns, and allow applicants to appeal decisions if they think the automated systems don't have full context.

---
# Section 5 — Reflection

*Answer in a few sentences each:*

1. **Prompting as engineering:** How is iterating on a prompt similar to and different from
   iterating on the model hyperparameters you tuned in Lab 3?
2. **Trust:** After your Section 4 evaluation, would you trust this system to run unattended?
   What single evaluation result most influenced your answer?
3. **Cost and scale:** Estimate (from your `response.usage` numbers) the tokens needed to
   process 1,000 applications per month. What does that imply for provider choice?
4. **Looking back at the course:** You have now used classical ML (Lab 2), trained neural
   networks (Lab 3), and used a foundation model via API (Lab 4). For a task like this one,
   why does calling an API beat training your own model — and when would it not?

> **Answer:** [Double-click to edit]

1. Prompting as engineering
Prompt iteration is similar to hyperparameter tuning because we test different configurations and evaluate their outputs to improve performance. The difference is that prompts change the instructions given to the LLM, while hyperparameters such as learning rate or epochs change how a model is trained.

2. I would not trust the system to run unattended because the evaluation showed that LLM outputs can vary and may hallucinate under adversarial inputs. The most important result was whether the model correctly handled missing or irrelevant information without fabricating values.

3. I would estimate the tokens for one application from the response.usage values and multiply the average by 1,000. So using L002,this application uses about 692 total tokens, 1,000 applications would require roughly 700,000 tokens per month. This means provider pricing like pay per use or subscription, free-tier limits, rate limits, and reliability should be considered when choosing an API to use.

4. Looking back at the course
For this task, calling an API is better because a foundation model is already trained on a huge amount of data, so we can build the system quickly without collecting a large dataset or spending resources training our own model. Training our own model would make more sense when we have large amounts of domain-specific data, strict privacy requirements, or a need for specialized behavior and control.

---
### Submission checklist

- [ ] All cells run top-to-bottom with no errors (`Kernel -> Restart & Run All`).
- [ ] **No API key anywhere in the notebook or the commit history.**
- [ ] Every **Student Reasoning** box is filled in with full sentences.
- [ ] `prompts.py` / `prompts.md` committed with your final prompt templates.
- [ ] Evaluation tables and adversarial test outputs visible in the saved notebook.
- [ ] Notebook pushed to `lab-4-llm-decision-support` with incremental commits.
- [ ] Repository link submitted to the course portal.
- [ ] AI Declaration form in Repository.